# Tutorial 9 (T9): Support Vector Machine (SVM)
**Goal:** Understand Linear SVM vs Non-Linear SVM, implement different kernels, and tune hyperparameter $C$.

## 💡 What is Linear vs Non-Linear SVM?
- **Linear SVM (`kernel='linear'`)**: Draws a **straight line** (or flat hyperplane in higher dimensions) to separate classes.
- **Non-Linear SVM (`kernel='rbf'`, `kernel='poly'` etc.)**: When data cannot be separated by a straight line (e.g. one class surrounds another in a circle), it uses the **Kernel Trick** to bend/curve the boundary.

### Step 1: Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

sns.set_theme(style="whitegrid")
print("Libraries loaded.")

### Step 2: Load Data & Standardize
Feature scaling is mandatory for SVM to calculate correct geometric margins.

In [ ]:
os.makedirs("data", exist_ok=True)
csv_path = os.path.join("data", "Crop_recommendation.csv")
if not os.path.exists(csv_path):
    crops = ["rice", "maize", "chickpea", "kidneybeans", "pigeonpeas", "mothbeans", "mungbean", "blackgram", "lentil", "pomegranate"]
    df = pd.DataFrame({
        'N': np.random.randint(10, 140, 200),
        'P': np.random.randint(5, 145, 200),
        'K': np.random.randint(15, 205, 200),
        'temperature': np.random.uniform(8.0, 43.0, 200),
        'humidity': np.random.uniform(14.0, 100.0, 200),
        'ph': np.random.uniform(3.5, 9.9, 200),
        'rainfall': np.random.uniform(20.0, 300.0, 200),
        'label': np.random.choice(crops, 200)
    })
    df.to_csv(csv_path, index=False)
else:
    df = pd.read_csv(csv_path)

X = df[['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)

### Step 3: Compare Linear vs Non-Linear SVM Kernels

In [ ]:
kernels = {
    'Linear (Straight Boundary)': SVC(kernel='linear', C=1.0),
    'Non-Linear RBF (Curved)': SVC(kernel='rbf', C=1.0, gamma='scale'),
    'Non-Linear Poly (degree=3)': SVC(kernel='poly', degree=3, C=1.0),
    'Non-Linear Sigmoid': SVC(kernel='sigmoid', C=1.0)
}

res = []
for name, m in kernels.items():
    m.fit(X_tr_sc, y_train)
    pred = m.predict(X_te_sc)
    res.append({
        'Model': name,
        'Accuracy (%)': accuracy_score(y_test, pred)*100,
        'F1-Score (%)': f1_score(y_test, pred, average='weighted', zero_division=0)*100
    })

res_df = pd.DataFrame(res)
display(res_df)

plt.figure(figsize=(7, 4))
plt.bar(res_df['Model'], res_df['Accuracy (%)'], color='#2b5c8f')
plt.xticks(rotation=20, ha='right')
plt.ylabel('Accuracy (%)')
plt.title('T9: Linear vs Non-Linear SVM Kernels')
plt.ylim(0, 105)
plt.show()

### Step 4: Hyperparameter $C$ Tuning
Small $C$ = wider margin (soft margin, prevents overfitting). Large $C$ = narrow margin (hard margin, strictly penalizes errors).

In [ ]:
c_vals = [0.01, 0.1, 1, 10, 100]
accs = []
for c in c_vals:
    m = SVC(kernel='rbf', C=c)
    m.fit(X_tr_sc, y_train)
    accs.append(accuracy_score(y_test, m.predict(X_te_sc))*100)

plt.figure(figsize=(6, 3.5))
plt.plot([str(c) for c in c_vals], accs, marker='o', color='#e76f51', linewidth=2)
plt.title('T9: Effect of Regularization C (RBF)')
plt.xlabel('C')
plt.ylabel('Accuracy (%)')
plt.show()

### Step 5: Confusion Matrix

In [ ]:
best_svm = SVC(kernel='rbf', C=10.0)
best_svm.fit(X_tr_sc, y_train)
pred_best = best_svm.predict(X_te_sc)

top_classes = sorted(list(set(y_test)))[:8]
cm = confusion_matrix(y_test, pred_best, labels=top_classes)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=top_classes, yticklabels=top_classes)
plt.title('T9: Confusion Matrix (RBF Kernel)')
plt.show()